# PK Used-Car Valuation — Exploratory Data Analysis

EDA over `data/raw/listings_full.parquet` (7,004 self-scraped PakWheels listings across
Karachi, Lahore, Islamabad, Rawalpindi). **This notebook is exploratory only** — it is
never the deliverable (SPEC §9). The split and leakage guard it motivates live as tested
`src/` code (`pkcar.data.split`, `pkcar.data.checks`).

## ⚠️ Critical caveat — asking price, not sale price (SPEC §3)

PakWheels prices are **seller-set asking prices**, not transaction prices, and they are
**biased upward**: sellers copy prices of similar listings, which anchors and inflates the
market. Everything below describes *asking* price. The model predicts asking price, and
the README states this openly — naming the bias and its direction is the point.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pkcar.data.checks import target_leakage_scores
from pkcar.data.loaders import LEAKAGE_COLS, TARGET_COL, feature_columns, load_listings
from pkcar.data.split import grouped_train_test_split, make_group_key

sns.set_theme(style="whitegrid")
FIGDIR = Path("../reports/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

# Load WITHOUT dropping missing target so we can see the raw missingness first.
raw = load_listings(drop_missing_target=False)
print(raw.shape)
raw.head(3)

## 1. Target distribution — why `log(price_pkr)` (SPEC §5)

Prices are right-skewed and multiplicative (a Mehran and a Land Cruiser differ by orders
of magnitude). We expect the log to be far more symmetric — which justifies training on
`log(price_pkr)` and reporting metrics back in raw PKR.

In [ ]:
price = raw[TARGET_COL].dropna()
summary = price.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
print(summary.round(0))
print(f"\nmissing target (Call-for-price): {raw[TARGET_COL].isna().sum()} "
      f"({raw[TARGET_COL].isna().mean():.1%})")
print(f"skew  raw: {price.skew():.3f}    skew log: {np.log(price).skew():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(price / 1e6, bins=60, ax=axes[0], color="#c0392b")
axes[0].set(title=f"Raw price (skew {price.skew():.2f})", xlabel="price (PKR, millions)")
sns.histplot(np.log(price), bins=60, ax=axes[1], color="#2471a3")
axes[1].set(title=f"log(price) (skew {np.log(price).skew():.2f})", xlabel="log(price_pkr)")
fig.suptitle("Log transform makes the target ~symmetric — justifies log target (SPEC §5)")
fig.tight_layout()
fig.savefig(FIGDIR / "target_log_transform.png", dpi=110, bbox_inches="tight")
plt.show()

**Decision:** raw skew ≈ 7.6 collapses to ≈ 0.08 under log — near-symmetric. We train on
`log(price_pkr)` (SPEC §5). The 114 "Call for price" listings have no target and are
dropped for modeling (`load_listings(drop_missing_target=True)`).

## 2. Missingness

In [ ]:
miss = (raw.isna().mean() * 100).round(2).sort_values(ascending=False)
miss = miss[miss > 0]
print(miss.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
miss.sort_values().plot.barh(ax=ax, color="#8e44ad")
ax.set(title="Missingness by column (%)", xlabel="% missing")
fig.savefig(FIGDIR / "missingness.png", dpi=110, bbox_inches="tight")
plt.show()

Missingness is low: target 1.6%, engine_cc 1.6%, the detail fields (`registration_city`,
`color`, `body_type`, `seller_type`) 0.6% — pages with no spec table. Imputation is done
**inside** the CV-fold pipeline (SPEC §5), never on the full frame.

## 3. Cardinality — make / model / registration_city (SPEC §4)

High-cardinality categoricals need native categorical handling (CatBoost/LightGBM), not
naive one-hot (SPEC §4). Registration city is a first-class feature, kept distinct from
listing city.

In [ ]:
for c in ["make", "model", "registration_city", "city", "body_type",
          "fuel", "transmission", "color", "seller_type"]:
    top = raw[c].value_counts().head(3).to_dict()
    print(f"{c:18s} nunique={raw[c].nunique():4d}   top={top}")

**Data-quality flags for feature engineering (Phase 4):**
- `make` 52, `model` 282, `registration_city` 23 — clean, native-categorical territory.
- `color` ≈ 490 levels ("White" vs "Solid White" vs "Pearl White") — needs normalization
  / rare-bucketing.
- `seller_type` ≈ 4,900 levels — the owner block over-captured text on some pages. Only a
  coarse **Dealer vs Individual** flag is trustworthy ("dealer" substring ≈ 2,050 rows);
  it will be collapsed to that binary, not used raw.

In [ ]:
# registration_city is genuinely distinct from listing city — it carries price signal.
reg = raw["registration_city"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(7, 4))
reg.sort_values().plot.barh(ax=ax, color="#16a085")
ax.set(title="Top registration cities (distinct from listing city, SPEC §4)", xlabel="listings")
fig.savefig(FIGDIR / "registration_city.png", dpi=110, bbox_inches="tight")
plt.show()

## 4. Leakage check — `canonical_price_pkr` is the target (SPEC §5)

The detail page repeats the price. It correlates ~perfectly with the target and is exactly
equal in ~89% of rows — a textbook leakage feature. It is declared in `LEAKAGE_COLS` and
excluded from `feature_columns()`; the pytest leakage guard fails if it (or anything that
encodes the target) is fed as a feature.

In [ ]:
m = raw[TARGET_COL].notna() & raw["canonical_price_pkr"].notna()
a, b = raw.loc[m, TARGET_COL], raw.loc[m, "canonical_price_pkr"]
print(f"corr(price, canonical) = {a.corr(b):.4f}   exact-equal = {(a == b).mean():.1%}")

model_df = load_listings(drop_missing_target=True)
# Leakage score of every declared feature (max legit feature ~0.87 = `model`)...
scores = target_leakage_scores(model_df, feature_columns(), TARGET_COL)
print("\nfeature leakage scores (max legit ~0.87):")
print(pd.Series(scores).sort_values(ascending=False).round(3).to_string())
# ...and the leaked column, for contrast:
leaked = target_leakage_scores(model_df, list(LEAKAGE_COLS), TARGET_COL)
score = leaked["canonical_price_pkr"]
print(f"\ncanonical_price_pkr leakage score = {score:.4f} (guard trips >= 0.98)")

## 5. Split strategy — grouped by vehicle, not random, not temporal (SPEC §5)

Popular models recur hundreds of times; a random split scatters a model's near-duplicate
listings across train and test and leaks its price level. We group by **make+model** so a
model lands entirely in train or entirely in test.

A *temporal* split is not available: this is a single snapshot with no listing-posted date
(SPEC §11); `scrape_date` only records when our resumable crawler ran. Grouped CV
(`GroupKFold`) is the honest, leakage-safe choice.

In [ ]:
groups = make_group_key(model_df)
gsize = groups.value_counts()
print(f"{gsize.size} vehicle groups; largest: {gsize.head(5).to_dict()}")
print(f"singletons (1 listing): {(gsize == 1).sum()}")

train_idx, test_idx = grouped_train_test_split(model_df, test_size=0.2, seed=42)
train_g, test_g = set(groups.iloc[train_idx]), set(groups.iloc[test_idx])
print(f"\ntrain rows {len(train_idx)}  test rows {len(test_idx)}")
print(f"shared vehicle groups between train & test: {len(train_g & test_g)}  (must be 0)")

## Decisions carried into Phase 4/5

1. **Target** = `log(price_pkr)`; metrics reported in raw PKR (MAE/RMSE/MAPE).
2. **Drop** the 114 no-price listings; they have no target.
3. **Exclude** `canonical_price_pkr` (leakage) — enforced by the pytest guard.
4. **Grouped CV** on make+model (`GroupKFold`); a held-out grouped test set for final
   reporting. No naive KFold, no `scrape_date` as a temporal axis.
5. **Clean up** `color` (normalize/rare-bucket) and collapse `seller_type` to
   Dealer/Individual in the feature layer.
6. All framing stays in **asking-price** terms (SPEC §3).